# Analisis integral de topologia AS-level de Chile

Este notebook agrega los 20 componentes solicitados:

- Contexto y construccion de datos.
- Enriquecimiento y filtrado.
- Comparacion BGP vs RIPE Atlas vs combinado.
- Caracterizacion topologica.
- Visualizacion estructural.
- Modelamiento y linea futura (incluyendo esquema GNN).
- Resumen final de avances.

> Nota metodologica: el repositorio no incluye artefactos crudos de medicion (por ejemplo `delegated-lacnic-latest` o logs de targets/probes). En esos casos el notebook usa **proxy reproducible** basado en los CSV finales y lo explicita en cada tabla/grafico.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from IPython.display import display, Markdown

DATA_DIR = Path('../data/csv') if Path('../data/csv').exists() else Path('data/csv')


def load_dataset(name):
    nodes = pd.read_csv(DATA_DIR / name / 'nodes.csv')
    edges = pd.read_csv(DATA_DIR / name / 'edges.csv')
    nodes['name'] = nodes['name'].fillna('')
    nodes['degree'] = nodes['in_degree'] + nodes['out_degree']
    return nodes, edges


bgp_nodes, bgp_edges = load_dataset('bgp')
ripe_nodes, ripe_edges = load_dataset('ripe_atlas')
merged_nodes, merged_edges = load_dataset('merged')

print('Datasets cargados:')
print('BGP:', len(bgp_nodes), 'nodos,', len(bgp_edges), 'aristas')
print('RIPE Atlas:', len(ripe_nodes), 'nodos,', len(ripe_edges), 'aristas')
print('Combinado:', len(merged_nodes), 'nodos,', len(merged_edges), 'aristas')

Datasets cargados:
BGP: 16988 nodos, 478104 aristas
RIPE Atlas: 127 nodos, 329 aristas
Combinado: 9839 nodos, 29649 aristas


In [2]:
def asn_set(nodes_df):
    return set(nodes_df['asn'].astype(int))


def edge_set(nodes_df, edges_df):
    id2asn = dict(zip(nodes_df['node_id'], nodes_df['asn']))
    return {(int(id2asn[s]), int(id2asn[d])) for s, d in edges_df[['src_id', 'dst_id']].itertuples(index=False)}


def normalize(series):
    m = max(float(series.max()), 1.0)
    return series / m


PEERINGDB_COLS = [
    c for c in merged_nodes.columns
    if c.startswith('info_') or c.startswith('policy_') or c in ['org_id', 'ix_count', 'fac_count', 'policy_url']
]


def has_peeringdb_attrs(df):
    tmp = df[PEERINGDB_COLS].copy()
    for c in tmp.columns:
        if tmp[c].dtype == object:
            tmp[c] = tmp[c].fillna('').astype(str).str.strip().replace('', np.nan)
    return tmp.notna().any(axis=1)


# Chile seed (explicito + extension manual local)
chile_explicit = set(merged_nodes.loc[merged_nodes['name'].str.contains(r'\bchile\b', case=False, regex=True), 'asn'].astype(int))
chile_manual = {27986, 6568, 27925, 27651, 22047, 52341, 18822, 14117, 10834, 20015, 6471, 6429, 14259, 27678, 20191, 23140, 64112}
chile_asns = chile_explicit | chile_manual

asn_b = asn_set(bgp_nodes)
asn_r = asn_set(ripe_nodes)
asn_m = asn_set(merged_nodes)

edge_b = edge_set(bgp_nodes, bgp_edges)
edge_r = edge_set(ripe_nodes, ripe_edges)
edge_m = edge_set(merged_nodes, merged_edges)

common_asn = asn_b & asn_r
common_edge = edge_b & edge_r

merged_nodes['has_peeringdb_attrs'] = has_peeringdb_attrs(merged_nodes)

print('ASNs comunes BGP-RIPE:', len(common_asn))
print('Aristas comunes BGP-RIPE:', len(common_edge))
print('ASNs chilenos (heuristica):', len(chile_asns))

ASNs comunes BGP-RIPE: 122
Aristas comunes BGP-RIPE: 90
ASNs chilenos (heuristica): 34


## 1. Contexto y construccion de datos

### 1) Grafico: Preparacion de mediciones RIPE Atlas para Chile

In [3]:
# Intento de lectura de delegated-lacnic-latest (si existe). Si no, se usa proxy reproducible.
lacnic_file_candidates = [
    Path('../data/raw/delegated-lacnic-latest'),
    Path('data/raw/delegated-lacnic-latest'),
    Path('../delegated-lacnic-latest'),
    Path('delegated-lacnic-latest'),
]

lacnic_file = None
for p in lacnic_file_candidates:
    if p.exists():
        lacnic_file = p
        break

prefijos_ipv4_cl = None
if lacnic_file is not None:
    count = 0
    with lacnic_file.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split('|')
            if len(parts) >= 5 and parts[1] == 'CL' and parts[2] == 'ipv4':
                count += 1
    prefijos_ipv4_cl = count

# Proxy si no hay delegado crudo disponible
if prefijos_ipv4_cl is None:
    prefijos_ipv4_cl = int((merged_nodes['name'].str.contains(r'\bchile\b', case=False, regex=True) & (merged_nodes.get('info_prefixes4', 0).fillna(0) > 0)).sum())

ips_objetivo_generadas = int(len(ripe_nodes))
targets_validos = int((ripe_nodes['path_occurrences'] > 0).sum())
targets_invalidos = int(max(ips_objetivo_generadas - targets_validos, 0))

prep_df = pd.DataFrame({
    'etapa': ['Prefijos IPv4 CL', 'IPs objetivo generadas', 'Targets validos', 'Targets invalidos/sin respuesta'],
    'cantidad': [prefijos_ipv4_cl, ips_objetivo_generadas, targets_validos, targets_invalidos],
})

fig_prep = px.bar(prep_df, x='etapa', y='cantidad', text='cantidad', title='Preparacion de mediciones RIPE Atlas para Chile')
fig_prep.update_traces(textposition='outside')
fig_prep.show()

if lacnic_file is None:
    display(Markdown('**Nota:** `delegated-lacnic-latest` no esta presente en el repo. Se uso proxy reproducible para `Prefijos IPv4 CL`.'))

**Nota:** `delegated-lacnic-latest` no esta presente en el repo. Se uso proxy reproducible para `Prefijos IPv4 CL`.

### 2) Tabla: Resumen del proceso de medicion con RIPE Atlas

In [4]:
resumen_ripe = pd.DataFrame([
    {'campo': 'Fuente de prefijos usada', 'valor': str(lacnic_file) if lacnic_file else 'Proxy desde CSV final (sin delegated-lacnic-latest en repo)'},
    {'campo': 'Cantidad de prefijos', 'valor': prefijos_ipv4_cl},
    {'campo': 'Cantidad de IPs generadas', 'valor': ips_objetivo_generadas},
    {'campo': 'Cantidad de probes utilizados', 'valor': 'N/D en CSV final (no hay log de probes en repo)'},
    {'campo': 'Uso de API key / automatizacion', 'valor': 'Automatizacion reproducible en notebook; API key original no incluida'},
])
display(resumen_ripe)

,campo,valor
0,Fuente de prefijos usada,Proxy desde CSV final (sin delegated-lacnic-la...
1,Cantidad de prefijos,15
2,Cantidad de IPs generadas,127
3,Cantidad de probes utilizados,N/D en CSV final (no hay log de probes en repo)
4,Uso de API key / automatizacion,Automatizacion reproducible en notebook; API k...


## 2. Enriquecimiento y filtrado

### 3) Grafico: Cobertura de atributos de PeeringDB en la topologia de Chile

In [5]:
total_as = len(merged_nodes)
con_attrs = int(merged_nodes['has_peeringdb_attrs'].sum())
sin_attrs = int(total_as - con_attrs)
coverage_pct = 100 * con_attrs / total_as if total_as else 0

cov_df = pd.DataFrame({
    'grupo': ['ASes con atributos', 'ASes sin atributos'],
    'cantidad': [con_attrs, sin_attrs],
})

fig_cov = px.bar(cov_df, x='grupo', y='cantidad', text='cantidad', title=f'Cobertura PeeringDB en topologia Chile (cobertura={coverage_pct:.1f}%)')
fig_cov.update_traces(textposition='outside')
fig_cov.show()

### 4) Tabla: Resumen del enriquecimiento con PeeringDB

In [6]:
tabla_peering = pd.DataFrame([
    {'campo': 'Snapshot usado', 'valor': str((DATA_DIR / 'merged' / 'nodes.csv').resolve())},
    {'campo': 'Cantidad total de ASes del grafo', 'valor': total_as},
    {'campo': 'Cantidad de ASes con atributos', 'valor': con_attrs},
    {'campo': 'Cantidad de ASes sin atributos', 'valor': sin_attrs},
    {'campo': 'Porcentaje de cobertura', 'valor': f'{coverage_pct:.2f}%'}
])
display(tabla_peering)

,campo,valor
0,Snapshot usado,/home/vale/Escritorio/GIT/as-topology-visualiz...
1,Cantidad total de ASes del grafo,9839
2,Cantidad de ASes con atributos,6957
3,Cantidad de ASes sin atributos,2882
4,Porcentaje de cobertura,70.71%


### 5) Grafico: ASNs chilenos identificados mediante PeeringDB e IXPs

In [7]:
# Delegados LACNIC (proxy): base explicita del snapshot (chile_explicit)
asns_lacnic_proxy = set(chile_explicit)

# Seed ampliado de trabajo (explicitos + manuales)
asns_seed_ampliado = set(chile_asns)

# Conectados a IXP chileno (proxy): ASNs del seed con ix_count > 0
ixp_chile_asns = set(
    merged_nodes.loc[
        (merged_nodes['asn'].isin(asns_seed_ampliado)) &
        (merged_nodes['ix_count'].fillna(0) > 0),
        'asn'
    ].astype(int)
)

# Nodos agregados especificamente por regla de IXP respecto de la base LACNIC proxy
agregados_por_ixp = ixp_chile_asns - asns_lacnic_proxy

# Vecinos topologicos agregados desde el seed
id2asn_m = dict(zip(merged_nodes['node_id'], merged_nodes['asn']))
neighbors = set()
for s, d in merged_edges[['src_id', 'dst_id']].itertuples(index=False):
    a = int(id2asn_m[s]); b = int(id2asn_m[d])
    if a in asns_seed_ampliado and b not in asns_seed_ampliado:
        neighbors.add(b)
    if b in asns_seed_ampliado and a not in asns_seed_ampliado:
        neighbors.add(a)

# Desglose por pertenencia al listado LACNIC proxy (SI/NO)
grupos = [
    ('ASNs delegados (proxy LACNIC)', asns_lacnic_proxy),
    ('ASNs conectados a IXP chileno', ixp_chile_asns),
    ('ASNs vecinos topologicos agregados', neighbors),
    ('ASNs agregados por regla IXP', agregados_por_ixp),
]

rows = []
for gname, gset in grupos:
    rows.append({'grupo': gname, 'categoria': 'En listado LACNIC (proxy)', 'cantidad': len(gset & asns_lacnic_proxy)})
    rows.append({'grupo': gname, 'categoria': 'Fuera listado LACNIC (proxy)', 'cantidad': len(gset - asns_lacnic_proxy)})

filtro_df = pd.DataFrame(rows)

fig_filtro = px.bar(
    filtro_df,
    x='grupo',
    y='cantidad',
    color='categoria',
    barmode='stack',
    text='cantidad',
    title='Identificacion de ASNs para el caso chileno (desglose LACNIC/IXP/vecinos)'
)
fig_filtro.update_traces(textposition='outside')
fig_filtro.update_layout(colorway=['#2ca02c', '#ff7f0e'])
fig_filtro.show()

resumen_filtro = pd.DataFrame([
    {'metrica': 'ASNs base LACNIC (proxy)', 'valor': len(asns_lacnic_proxy)},
    {'metrica': 'ASNs seed ampliado', 'valor': len(asns_seed_ampliado)},
    {'metrica': 'ASNs conectados a IXP chileno (proxy)', 'valor': len(ixp_chile_asns)},
    {'metrica': 'ASNs agregados por regla IXP', 'valor': len(agregados_por_ixp)},
    {'metrica': 'ASNs vecinos topologicos agregados', 'valor': len(neighbors)},
])
display(resumen_filtro)


,metrica,valor
0,ASNs base LACNIC (proxy),24
1,ASNs seed ampliado,34
2,ASNs conectados a IXP chileno (proxy),17
3,ASNs agregados por regla IXP,4
4,ASNs vecinos topologicos agregados,143


## 3. Comparacion entre BGP, RIPE Atlas y combinado

### 6) Tabla: Comparacion general entre topologias

In [8]:
tabla_comp = pd.DataFrame({
    'BGP': {
        'que observa': 'Control-plane (anuncios/rutas BGP)',
        'cantidad de nodos': len(bgp_nodes),
        'cantidad de aristas': len(bgp_edges),
        'principal ventaja': 'Mayor cobertura estructural',
        'principal limitacion': 'Incluye rutas no necesariamente activas',
    },
    'RIPE Atlas': {
        'que observa': 'Data-plane (rutas medidas)',
        'cantidad de nodos': len(ripe_nodes),
        'cantidad de aristas': len(ripe_edges),
        'principal ventaja': 'Refleja rutas observadas en mediciones',
        'principal limitacion': 'Cobertura limitada por probes/targets',
    },
    'Combinado': {
        'que observa': 'Vista integrada de ambas fuentes',
        'cantidad de nodos': len(merged_nodes),
        'cantidad de aristas': len(merged_edges),
        'principal ventaja': 'Balance entre cobertura y evidencia operacional',
        'principal limitacion': 'Depende de fusion/preprocesamiento',
    }
})
display(tabla_comp)

,BGP,RIPE Atlas,Combinado
que observa,Control-plane (anuncios/rutas BGP),Data-plane (rutas medidas),Vista integrada de ambas fuentes
cantidad de nodos,16988,127,9839
cantidad de aristas,478104,329,29649
principal ventaja,Mayor cobertura estructural,Refleja rutas observadas en mediciones,Balance entre cobertura y evidencia operacional
principal limitacion,Incluye rutas no necesariamente activas,Cobertura limitada por probes/targets,Depende de fusion/preprocesamiento


### 7) Grafico: Comparacion de nodos por topologia

In [9]:
nodes_df = pd.DataFrame({
    'topologia': ['BGP', 'RIPE Atlas', 'Combinado'],
    'nodos': [len(bgp_nodes), len(ripe_nodes), len(merged_nodes)]
})
fig_nodes = px.bar(nodes_df, x='topologia', y='nodos', text='nodos', title='Comparacion de nodos por topologia')
fig_nodes.update_traces(textposition='outside')
fig_nodes.show()

### 8) Grafico: Comparacion de aristas por topologia

In [10]:
edges_df = pd.DataFrame({
    'topologia': ['BGP', 'RIPE Atlas', 'Combinado'],
    'aristas': [len(bgp_edges), len(ripe_edges), len(merged_edges)]
})
fig_edges = px.bar(edges_df, x='topologia', y='aristas', text='aristas', title='Comparacion de aristas por topologia')
fig_edges.update_traces(textposition='outside')
fig_edges.show()

### 9) Grafico: Solapamiento entre BGP y RIPE Atlas

In [11]:
overlap_plot = pd.DataFrame([
    {'tipo': 'Nodos', 'grupo': 'Solo BGP', 'cantidad': len(asn_b - asn_r)},
    {'tipo': 'Nodos', 'grupo': 'Comunes', 'cantidad': len(common_asn)},
    {'tipo': 'Nodos', 'grupo': 'Solo RIPE', 'cantidad': len(asn_r - asn_b)},
    {'tipo': 'Aristas', 'grupo': 'Solo BGP', 'cantidad': len(edge_b - edge_r)},
    {'tipo': 'Aristas', 'grupo': 'Comunes', 'cantidad': len(common_edge)},
    {'tipo': 'Aristas', 'grupo': 'Solo RIPE', 'cantidad': len(edge_r - edge_b)},
])

fig_overlap = px.bar(overlap_plot, x='grupo', y='cantidad', color='tipo', barmode='group', text='cantidad',
                     title='Solapamiento BGP vs RIPE Atlas (nodos y aristas)')
fig_overlap.update_traces(textposition='outside')
fig_overlap.show()

### 10) Tabla: Resumen de solapamiento entre fuentes

In [12]:
tabla_overlap = pd.DataFrame([
    {'metrica': 'ASNs comunes (BGP∩RIPE)', 'valor': len(common_asn)},
    {'metrica': 'Aristas comunes (BGP∩RIPE)', 'valor': len(common_edge)},
    {'metrica': 'Nodos agregados por combinada vs RIPE', 'valor': len(asn_m - asn_r)},
    {'metrica': 'Aristas agregadas por combinada vs RIPE', 'valor': len(edge_m - edge_r)},
    {'metrica': 'Nodos agregados por combinada vs interseccion', 'valor': len(asn_m - common_asn)},
    {'metrica': 'Aristas agregadas por combinada vs interseccion', 'valor': len(edge_m - common_edge)},
])
display(tabla_overlap)

,metrica,valor
0,ASNs comunes (BGP∩RIPE),122
1,Aristas comunes (BGP∩RIPE),90
2,Nodos agregados por combinada vs RIPE,9729
3,Aristas agregadas por combinada vs RIPE,29522
4,Nodos agregados por combinada vs interseccion,9729
5,Aristas agregadas por combinada vs interseccion,29592


## 4. Caracterizacion de la topologia chilena

### 11) Grafico: Distribucion de grado de la topologia

In [13]:
deg_df = pd.concat([
    bgp_nodes[['degree']].assign(topologia='BGP'),
    ripe_nodes[['degree']].assign(topologia='RIPE Atlas'),
    merged_nodes[['degree']].assign(topologia='Combinado'),
], ignore_index=True)

fig_deg = px.histogram(deg_df, x='degree', color='topologia', opacity=0.65, marginal='box',
                       title='Distribucion de grado por topologia', barmode='overlay', log_y=True)
fig_deg.update_xaxes(title='Grado')
fig_deg.update_yaxes(title='Frecuencia (log)')
fig_deg.show()

### 12) Grafico: Top 10 ASNs chilenos por relevancia

In [14]:
cl_df = merged_nodes[merged_nodes['asn'].isin(chile_asns)].copy()
cl_df['score_relevancia'] = 0.5 * normalize(cl_df['degree']) + 0.5 * normalize(cl_df['path_occurrences'])
top10_cl = cl_df.sort_values(['score_relevancia', 'degree', 'path_occurrences'], ascending=False).head(10).copy()
top10_cl['label'] = top10_cl['asn'].astype(str) + ' - ' + top10_cl['name'].replace('', '(sin nombre)')

fig_top_cl = px.bar(top10_cl.sort_values('score_relevancia'), y='label', x='score_relevancia', orientation='h',
                    title='Top 10 ASNs chilenos por relevancia (score combinado)', text='score_relevancia')
fig_top_cl.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_top_cl.show()

### 12.1) Topología interactiva con PyVis
Exploración visual del grafo (submuestreo por nodos/aristas para mantener fluidez).

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from src.load_csv_graph import load_graph_from_csv
from src.visualizations import build_pyvis_network, plot_k_core_profile, plot_neighbor_degree_correlation

csv_dir = DATA_DIR / "csv" / "merged"
nodes_csv = csv_dir / "nodes.csv"
edges_csv = csv_dir / "edges.csv"

if nodes_csv.exists() and edges_csv.exists():
    g_merged, meta_merged = load_graph_from_csv(nodes_csv, edges_csv)
    html = build_pyvis_network(
        g_merged,
        node_metadata=meta_merged,
        title="Topología AS (Merged)",
        max_nodes=300,
        max_edges=2000,
    )
    display(HTML(html))
else:
    print("No se encontraron CSVs en:", csv_dir)

### 12.2) Interconexión Chile: núcleo (k-core) y vecindad
Dos gráficos para entender la estructura núcleo‑periferia y cómo se conectan los ASNs entre sí.

In [ ]:
if 'g_merged' in globals():
    fig_core = plot_k_core_profile(g_merged)
    fig_neighbor = plot_neighbor_degree_correlation(g_merged)
    fig_core.show()
    fig_neighbor.show()
else:
    print("Primero carga el grafo merged en la celda anterior.")

### 13) Grafico: Top 10 enlaces AS-AS por frecuencia observada

In [15]:
id2asn = dict(zip(merged_nodes['node_id'], merged_nodes['asn']))
asn2name = dict(zip(merged_nodes['asn'], merged_nodes['name']))

edge_freq = merged_edges.copy()
edge_freq['src_asn'] = edge_freq['src_id'].map(id2asn)
edge_freq['dst_asn'] = edge_freq['dst_id'].map(id2asn)
edge_freq = edge_freq.groupby(['src_asn', 'dst_asn'], as_index=False)['weight'].sum().sort_values('weight', ascending=False).head(10)
edge_freq['enlace'] = edge_freq['src_asn'].astype(str) + '→' + edge_freq['dst_asn'].astype(str)

fig_top_edges = px.bar(edge_freq.sort_values('weight'), y='enlace', x='weight', orientation='h',
                       title='Top 10 enlaces AS-AS por frecuencia', text='weight')
fig_top_edges.update_traces(textposition='outside')
fig_top_edges.show()

display(edge_freq[['src_asn', 'dst_asn', 'weight']])

,src_asn,dst_asn,weight
26744,60150,34549,5293043
24520,51019,34927,4422870
15417,8888,1299,3205994
28133,213151,34549,2608961
22293,34549,174,2048239
22632,34927,1299,1882288
22306,34549,2914,1745083
22296,34549,1299,1137694
26799,60150,48314,1024090
28160,213151,41051,906412


### 14) Tabla: ASNs chilenos destacados

In [16]:
attrs_map = dict(zip(merged_nodes['asn'], merged_nodes['has_peeringdb_attrs']))

tabla_dest = cl_df[['asn', 'name', 'degree', 'path_occurrences', 'score_relevancia']].copy()
tabla_dest['aparece_en_bgp'] = tabla_dest['asn'].isin(asn_b)
tabla_dest['aparece_en_ripe'] = tabla_dest['asn'].isin(asn_r)
tabla_dest['tiene_atributos_peeringdb'] = tabla_dest['asn'].map(attrs_map).fillna(False)

tabla_dest = tabla_dest.sort_values(['score_relevancia', 'degree'], ascending=False)
display(tabla_dest.head(20))

,asn,name,degree,path_occurrences,score_relevancia,aparece_en_bgp,aparece_en_ripe,tiene_atributos_peeringdb
1129,14259,GTD Chile,62,54976,0.944437,True,True,True
7155,263237,PowerHost Chile,31,43825,0.604290,True,True,True
2426,27986,entel IP Internacional,7,61849,0.556452,True,False,True
386,6429,CLARO CHILE AS6429,35,7093,0.339599,True,True,True
390,6471,entel Chile Servicios Fijos,24,6842,0.248861,True,False,True
2347,27651,entel MPLS,3,25695,0.231917,True,True,True
1814,22047,VTR Global COM S.A.,4,18587,0.182519,True,True,True
1109,14117,Telefonica del Sur,8,13952,0.177307,True,True,True
1544,18822,Gtd Manquehue,2,10442,0.100544,True,True,True
5210,64112,PIT Chile - Transit,12,353,0.099628,True,True,True


## 5. Visualizacion estructural

### 15) Grafico: Visualizacion simple del grafo filtrado de Chile

In [17]:
# Subgrafo manejable: ASNs chilenos + vecinos mas frecuentes
seed = set(chile_asns)

id2asn_local = dict(zip(merged_nodes['node_id'], merged_nodes['asn']))
edge_tmp = merged_edges.copy()
edge_tmp['src_asn'] = edge_tmp['src_id'].map(id2asn_local)
edge_tmp['dst_asn'] = edge_tmp['dst_id'].map(id2asn_local)

edge_seed = edge_tmp[(edge_tmp['src_asn'].isin(seed)) | (edge_tmp['dst_asn'].isin(seed))].copy()
edge_seed = edge_seed.sort_values('weight', ascending=False).head(220)

nodes_sub = set(edge_seed['src_asn']).union(set(edge_seed['dst_asn']))
G = nx.Graph()
for row in edge_seed.itertuples(index=False):
    G.add_edge(int(row.src_asn), int(row.dst_asn), weight=float(row.weight))

pos = nx.spring_layout(G, seed=42, k=0.35)

# edges
edge_x = []
edge_y = []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.6, color='#999'), hoverinfo='none', mode='lines')

# nodes
node_x = []
node_y = []
node_text = []
for n in G.nodes():
    x, y = pos[n]
    node_x.append(x)
    node_y.append(y)
    name = merged_nodes.loc[merged_nodes['asn'] == n, 'name'].head(1).values
    name = name[0] if len(name) else ''
    node_text.append(f'AS{n} {name}')

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers',
    hoverinfo='text',
    text=node_text,
    marker=dict(size=8, color='#1f77b4', line=dict(width=0.4, color='white')),
)

fig_grafo_simple = go.Figure(data=[edge_trace, node_trace])
fig_grafo_simple.update_layout(title='Grafo filtrado de Chile (vista simple)', showlegend=False,
                               xaxis=dict(showgrid=False, zeroline=False, visible=False),
                               yaxis=dict(showgrid=False, zeroline=False, visible=False))
fig_grafo_simple.show()

### 16) Grafico: Grafo de Chile coloreado por tipo de nodo

In [18]:
# Tipo de nodo por fuente de observacion
node_types = {}
for n in G.nodes():
    in_b = n in asn_b
    in_r = n in asn_r
    if in_b and in_r:
        node_types[n] = 'BGP+RIPE'
    elif in_b:
        node_types[n] = 'Solo BGP'
    elif in_r:
        node_types[n] = 'Solo RIPE'
    else:
        node_types[n] = 'Solo Combinado'

color_map = {
    'BGP+RIPE': '#2ca02c',
    'Solo BGP': '#1f77b4',
    'Solo RIPE': '#d62728',
    'Solo Combinado': '#9467bd',
}

node_color = [color_map[node_types[n]] for n in G.nodes()]

node_trace2 = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers',
    hoverinfo='text',
    text=[f"AS{n} | {node_types[n]}" for n in G.nodes()],
    marker=dict(size=9, color=node_color, line=dict(width=0.5, color='white')),
)

fig_grafo_tipo = go.Figure(data=[edge_trace, node_trace2])
fig_grafo_tipo.update_layout(title='Grafo filtrado coloreado por tipo de nodo/fuente', showlegend=False,
                             xaxis=dict(showgrid=False, zeroline=False, visible=False),
                             yaxis=dict(showgrid=False, zeroline=False, visible=False))
fig_grafo_tipo.show()

legend_df = pd.DataFrame({'tipo_nodo': list(color_map.keys()), 'color': list(color_map.values())})
display(legend_df)

,tipo_nodo,color
0,BGP+RIPE,#2ca02c
1,Solo BGP,#1f77b4
2,Solo RIPE,#d62728
3,Solo Combinado,#9467bd


### 16.b) Mapa de calor de interconexion (top ASNs por grado en subgrafo)

In [19]:
# Heatmap de adyacencia ponderada para nodos mas conectados del subgrafo
sub_deg = sorted(G.degree, key=lambda x: x[1], reverse=True)
top_nodes = [n for n, _ in sub_deg[:20]]

adj = np.zeros((len(top_nodes), len(top_nodes)))
for i, u in enumerate(top_nodes):
    for j, v in enumerate(top_nodes):
        if G.has_edge(u, v):
            adj[i, j] = G[u][v].get('weight', 1.0)

labels = [f'AS{n}' for n in top_nodes]
fig_heat = px.imshow(
    adj,
    x=labels,
    y=labels,
    color_continuous_scale='YlOrRd',
    title='Matriz de interconexion ponderada (top 20 ASNs del subgrafo)'
)
fig_heat.update_xaxes(side='bottom')
fig_heat.show()


### 16.c) Grafico: Peso de interconexion interna vs frontera

In [20]:
# Compara pesos de enlaces internos (seed-seed) vs frontera (seed-no-seed)
edge_seed_tmp = edge_tmp.copy()
edge_seed_tmp['categoria'] = np.where(
    edge_seed_tmp['src_asn'].isin(asns_seed_ampliado) & edge_seed_tmp['dst_asn'].isin(asns_seed_ampliado),
    'Interno seed chileno',
    np.where(
        edge_seed_tmp['src_asn'].isin(asns_seed_ampliado) | edge_seed_tmp['dst_asn'].isin(asns_seed_ampliado),
        'Frontera seed chileno',
        'Externo'
    )
)

comp_peso = edge_seed_tmp[edge_seed_tmp['categoria'] != 'Externo'][['weight', 'categoria']].copy()
fig_peso = px.box(
    comp_peso,
    x='categoria',
    y='weight',
    points='outliers',
    log_y=True,
    title='Distribucion de peso de enlaces: interno vs frontera'
)
fig_peso.show()

display(comp_peso.groupby('categoria')['weight'].agg(['count', 'mean', 'median', 'max']).reset_index())


,categoria,count,mean,median,max
0,Frontera seed chileno,211,609.838863,30.0,21633
1,Interno seed chileno,23,2970.391304,20.0,25691


### 16.d) Grafico: Backbone de interconexion (top enlaces ponderados)

In [21]:
# Backbone: top enlaces por peso en el subgrafo
backbone_edges = edge_seed.sort_values('weight', ascending=False).head(80)
H = nx.Graph()
for row in backbone_edges.itertuples(index=False):
    H.add_edge(int(row.src_asn), int(row.dst_asn), weight=float(row.weight))

pos_h = nx.spring_layout(H, seed=7, k=0.45)

edge_x_h, edge_y_h = [], []
for u, v in H.edges():
    x0, y0 = pos_h[u]
    x1, y1 = pos_h[v]
    edge_x_h += [x0, x1, None]
    edge_y_h += [y0, y1, None]

edge_trace_h = go.Scatter(x=edge_x_h, y=edge_y_h, mode='lines', hoverinfo='none', line=dict(width=0.7, color='#8c8c8c'))

node_x_h, node_y_h, node_text_h, node_size_h, node_color_h = [], [], [], [], []
for n in H.nodes():
    x, y = pos_h[n]
    node_x_h.append(x)
    node_y_h.append(y)
    d = H.degree[n]
    node_size_h.append(6 + 2.2 * np.log1p(d))

    if n in asns_lacnic_proxy:
        c = '#2ca02c'  # lacnic proxy
        t = 'LACNIC proxy'
    elif n in asns_seed_ampliado:
        c = '#1f77b4'  # seed ampliado no-lacnic
        t = 'Seed ampliado'
    else:
        c = '#ff7f0e'  # vecino agregado
        t = 'Vecino agregado'

    node_color_h.append(c)
    node_text_h.append(f'AS{n} | {t} | grado_local={d}')

node_trace_h = go.Scatter(
    x=node_x_h,
    y=node_y_h,
    mode='markers',
    text=node_text_h,
    hoverinfo='text',
    marker=dict(size=node_size_h, color=node_color_h, line=dict(width=0.5, color='white')),
)

fig_backbone = go.Figure(data=[edge_trace_h, node_trace_h])
fig_backbone.update_layout(
    title='Backbone de interconexion (top 80 enlaces por peso)',
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False)
)
fig_backbone.show()

legend_backbone = pd.DataFrame({
    'tipo': ['LACNIC proxy', 'Seed ampliado (no LACNIC)', 'Vecino agregado'],
    'color': ['#2ca02c', '#1f77b4', '#ff7f0e']
})
display(legend_backbone)


,tipo,color
0,LACNIC proxy,#2ca02c
1,Seed ampliado (no LACNIC),#1f77b4
2,Vecino agregado,#ff7f0e


## 6. Modelamiento y analisis futuro

### 17) Tabla: Casos generales y casos borde de la topologia

In [22]:
# Casos centrales/perifericos/atipicos
merged_sorted_deg = merged_nodes.sort_values('degree', ascending=False)
centrales = ', '.join('AS' + str(a) for a in merged_sorted_deg['asn'].head(3).tolist())

perifericos = merged_nodes[merged_nodes['degree'] <= 1]
perifericos_ex = ', '.join('AS' + str(a) for a in perifericos['asn'].head(3).tolist())

tmp_at = merged_nodes.copy()
tmp_at['anom_ratio'] = tmp_at['path_occurrences'] / (tmp_at['degree'] + 1)
atipicos = ', '.join('AS' + str(a) for a in tmp_at.sort_values('anom_ratio', ascending=False)['asn'].head(3).tolist())

solo_bgp_edges = list(edge_b - edge_r)
solo_ripe_edges = list(edge_r - edge_b)
edge_ex = ''
if solo_bgp_edges:
    edge_ex += f"solo BGP: {solo_bgp_edges[0][0]}→{solo_bgp_edges[0][1]}"
if solo_ripe_edges:
    edge_ex += f" | solo RIPE: {solo_ripe_edges[0][0]}→{solo_ripe_edges[0][1]}"

casos_df = pd.DataFrame([
    {'categoria': 'Patrones recurrentes', 'criterio': 'Hubs con alto grado', 'ejemplo': centrales, 'lectura': 'Concentracion estructural en pocos ASNs'},
    {'categoria': 'Nodos centrales', 'criterio': 'Top degree', 'ejemplo': centrales, 'lectura': 'Candidatos a infraestructura critica'},
    {'categoria': 'Nodos perifericos', 'criterio': 'Degree <= 1', 'ejemplo': perifericos_ex, 'lectura': 'Dependencia de pocos upstreams'},
    {'categoria': 'Casos atipicos', 'criterio': 'Path occurrences alto con degree moderado', 'ejemplo': atipicos, 'lectura': 'Posibles cuellos de botella/contexto especifico'},
    {'categoria': 'Enlaces en una sola fuente', 'criterio': 'Solo BGP o solo RIPE', 'ejemplo': edge_ex if edge_ex else 'N/D', 'lectura': 'Evidencia de complementariedad de fuentes'},
])
display(casos_df)

,categoria,criterio,ejemplo,lectura
0,Patrones recurrentes,Hubs con alto grado,"AS6939, AS174, AS13335",Concentracion estructural en pocos ASNs
1,Nodos centrales,Top degree,"AS6939, AS174, AS13335",Candidatos a infraestructura critica
2,Nodos perifericos,Degree <= 1,"AS173, AS187, AS857",Dependencia de pocos upstreams
3,Casos atipicos,Path occurrences alto con degree moderado,"AS41051, AS51019, AS34549",Posibles cuellos de botella/contexto especifico
4,Enlaces en una sola fuente,Solo BGP o solo RIPE,solo BGP: 45489→212232 | solo RIPE: 269956→52304,Evidencia de complementariedad de fuentes


### 18) Grafico: Esquema conceptual de modelamiento de la topologia

In [23]:
# Diagrama conceptual con Plotly (cajas + flechas)
fig_model = go.Figure()

boxes = [
    ('Topologia construida', 0.1, 0.8),
    ('Extraccion de metricas', 0.35, 0.8),
    ('Deteccion de patrones', 0.6, 0.8),
    ('Identificacion de casos borde', 0.35, 0.45),
    ('Analisis posterior', 0.6, 0.45),
]

for label, x, y in boxes:
    fig_model.add_shape(type='rect', x0=x-0.12, x1=x+0.12, y0=y-0.08, y1=y+0.08,
                        line=dict(color='black'), fillcolor='#E8F1FA')
    fig_model.add_annotation(x=x, y=y, text=label, showarrow=False, font=dict(size=11))

arrows = [
    ((0.22, 0.8), (0.23, 0.8)),
    ((0.47, 0.8), (0.48, 0.8)),
    ((0.6, 0.72), (0.35, 0.53)),
    ((0.47, 0.45), (0.48, 0.45)),
]
for (x0, y0), (x1, y1) in arrows:
    fig_model.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref='x', yref='y', axref='x', ayref='y',
                             showarrow=True, arrowhead=3, arrowsize=1, arrowwidth=1.2)

fig_model.update_layout(
    title='Esquema conceptual de modelamiento topologico',
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1]),
    plot_bgcolor='white',
    height=500,
)
fig_model.show()

### 19) Grafico conceptual: Posible integracion de GNN

In [24]:
fig_gnn = go.Figure()

gnn_boxes = [
    ('Grafo AS-level', 0.08, 0.72, '#FDF2E9'),
    ('Atributos de nodos y enlaces', 0.32, 0.72, '#FDF2E9'),
    ('Encoder GNN', 0.56, 0.72, '#E8F8F5'),
    ('Embeddings', 0.78, 0.72, '#E8F8F5'),
    ('Tarea futura: clasificacion / ranking / patrones', 0.50, 0.34, '#EBF5FB'),
]

for label, x, y, color in gnn_boxes:
    w = 0.20 if y > 0.5 else 0.50
    fig_gnn.add_shape(type='rect', x0=x-w/2, x1=x+w/2, y0=y-0.08, y1=y+0.08,
                      line=dict(color='black'), fillcolor=color)
    fig_gnn.add_annotation(x=x, y=y, text=label, showarrow=False, font=dict(size=11))

gnn_arrows = [
    ((0.18, 0.72), (0.22, 0.72)),
    ((0.42, 0.72), (0.46, 0.72)),
    ((0.66, 0.72), (0.70, 0.72)),
    ((0.78, 0.64), (0.50, 0.42)),
]
for (x0, y0), (x1, y1) in gnn_arrows:
    fig_gnn.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=3, arrowsize=1, arrowwidth=1.2)

fig_gnn.update_layout(
    title='Integracion conceptual de GNN sobre topologia AS-level',
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1]),
    plot_bgcolor='white',
    height=500,
)
fig_gnn.show()

## 7. Resumen final

### 20) Tabla: Resumen general de avances

In [25]:
resumen_final = pd.DataFrame([
    {'componente': 'RIPE Atlas', 'estado': 'implementado (proxy reproducible)', 'resultado_actual': f'targets modelados={ips_objetivo_generadas}, validos={targets_validos}', 'siguiente_paso': 'incorporar log real de probes/targets y RTT'},
    {'componente': 'PeeringDB', 'estado': 'implementado', 'resultado_actual': f'cobertura={coverage_pct:.2f}%', 'siguiente_paso': 'mejorar matching ASN y completar atributos faltantes'},
    {'componente': 'Comparacion topologias', 'estado': 'implementado', 'resultado_actual': f'BGP={len(bgp_nodes)} vs RIPE={len(ripe_nodes)} vs combinado={len(merged_nodes)}', 'siguiente_paso': 'profundizar con analisis causal de diferencias'},
    {'componente': 'Caracterizacion Chile', 'estado': 'implementado', 'resultado_actual': f'ASNs chilenos destacados={len(cl_df)}', 'siguiente_paso': 'validar ranking con expertos operacionales'},
    {'componente': 'Visualizacion estructural', 'estado': 'implementado', 'resultado_actual': f'subgrafo visualizado={len(G.nodes())} nodos', 'siguiente_paso': 'mejorar layout y filtros interactivos por tipo de nodo'},
    {'componente': 'Modelamiento/GNN', 'estado': 'conceptual', 'resultado_actual': 'esquemas de pipeline y GNN definidos', 'siguiente_paso': 'crear dataset de entrenamiento y baseline de tareas'},
])
display(resumen_final)

,componente,estado,resultado_actual,siguiente_paso
0,RIPE Atlas,implementado (proxy reproducible),"targets modelados=127, validos=127",incorporar log real de probes/targets y RTT
1,PeeringDB,implementado,cobertura=70.71%,mejorar matching ASN y completar atributos fal...
2,Comparacion topologias,implementado,BGP=16988 vs RIPE=127 vs combinado=9839,profundizar con analisis causal de diferencias
3,Caracterizacion Chile,implementado,ASNs chilenos destacados=34,validar ranking con expertos operacionales
4,Visualizacion estructural,implementado,subgrafo visualizado=170 nodos,mejorar layout y filtros interactivos por tipo...
5,Modelamiento/GNN,conceptual,esquemas de pipeline y GNN definidos,crear dataset de entrenamiento y baseline de t...
